# 02 EDA and Unsupervised Learning

**Project:** STAT GR5243 Project 4: End-to-End Machine Learning  
**Topic:** Predicting High-Occupancy Airbnb Listings in New York City

This notebook performs exploratory data analysis and unsupervised learning on the cleaned Airbnb NYC dataset. It should be run after `01_data_loading_cleaning.ipynb`.


## 1. Import Libraries and Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

processed_path = Path("../data/processed")
fig_path = Path("../figures")
fig_path.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(processed_path / "nyc_airbnb_cleaned.csv", low_memory=False)

print("Cleaned data shape:", df.shape)
df.head()


## 2. Target Variable Distribution

The target variable is `high_occupancy`, where a listing is coded as 1 if `estimated_occupancy_l365d >= 60`.


In [ ]:
target_counts = df["high_occupancy"].value_counts().sort_index()
target_share = target_counts / target_counts.sum()

target_summary = pd.DataFrame({
    "count": target_counts,
    "share": target_share
})

target_summary


In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(target_share.index.astype(str), target_share.values)

plt.title("Share of Low vs High Occupancy Listings")
plt.xlabel("High Occupancy Indicator")
plt.ylabel("Share of Listings")
plt.xticks([0, 1], ["0 = Low/Regular", "1 = High"])

for i, v in enumerate(target_share.values):
    plt.text(i, v + 0.01, f"{v:.1%}", ha="center")

plt.tight_layout()
plt.savefig(fig_path / "01_target_distribution.png", dpi=200)
plt.show()


## 3. Borough-Level EDA

We compare high-occupancy rates across NYC boroughs.


In [ ]:
borough_summary = (
    df.groupby("neighbourhood_group_cleansed")
    .agg(
        listings=("id", "count"),
        high_occupancy_rate=("high_occupancy", "mean"),
        median_occupancy=("estimated_occupancy_l365d", "median"),
        avg_availability_365=("availability_365", "mean"),
        median_reviews=("number_of_reviews", "median")
    )
    .reset_index()
    .sort_values("high_occupancy_rate", ascending=False)
)

borough_summary


In [ ]:
plot_data = borough_summary.sort_values("high_occupancy_rate", ascending=True)

plt.figure(figsize=(8, 5))

plt.barh(
    plot_data["neighbourhood_group_cleansed"],
    plot_data["high_occupancy_rate"]
)

plt.title("High-Occupancy Rate by NYC Borough")
plt.xlabel("High-Occupancy Rate")
plt.ylabel("Borough")

for i, v in enumerate(plot_data["high_occupancy_rate"]):
    plt.text(v + 0.005, i, f"{v:.1%}", va="center")

plt.tight_layout()
plt.savefig(fig_path / "02_borough_high_occupancy_rate.png", dpi=200)
plt.show()


## 4. Room Type EDA

We compare occupancy patterns by room type.


In [ ]:
room_summary = (
    df.groupby("room_type")
    .agg(
        listings=("id", "count"),
        high_occupancy_rate=("high_occupancy", "mean"),
        median_occupancy=("estimated_occupancy_l365d", "median"),
        median_minimum_nights=("minimum_nights", "median"),
        median_reviews=("number_of_reviews", "median")
    )
    .reset_index()
    .sort_values("high_occupancy_rate", ascending=False)
)

room_summary


In [ ]:
plot_data = room_summary.sort_values("high_occupancy_rate", ascending=True)

plt.figure(figsize=(8, 5))

plt.barh(
    plot_data["room_type"],
    plot_data["high_occupancy_rate"]
)

plt.title("High-Occupancy Rate by Room Type")
plt.xlabel("High-Occupancy Rate")
plt.ylabel("Room Type")

for i, v in enumerate(plot_data["high_occupancy_rate"]):
    plt.text(v + 0.005, i, f"{v:.1%}", va="center")

plt.tight_layout()
plt.savefig(fig_path / "03_room_type_high_occupancy_rate.png", dpi=200)
plt.show()


## 5. Comparing High- and Low-Occupancy Listings

The plots below compare availability and review frequency across occupancy groups.


In [ ]:
plt.figure(figsize=(7, 5))

plt.boxplot(
    [
        df.loc[df["high_occupancy"] == 0, "availability_365"].dropna(),
        df.loc[df["high_occupancy"] == 1, "availability_365"].dropna()
    ],
    tick_labels=["Low/Regular", "High"],
    showfliers=False
)

plt.title("Availability 365 by Occupancy Group")
plt.xlabel("Occupancy Group")
plt.ylabel("Availability in Next 365 Days")

plt.tight_layout()
plt.savefig(fig_path / "04_availability_by_occupancy_group.png", dpi=200)
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))

plt.boxplot(
    [
        df.loc[df["high_occupancy"] == 0, "reviews_per_month"].dropna(),
        df.loc[df["high_occupancy"] == 1, "reviews_per_month"].dropna()
    ],
    tick_labels=["Low/Regular", "High"],
    showfliers=False
)

plt.title("Reviews per Month by Occupancy Group")
plt.xlabel("Occupancy Group")
plt.ylabel("Reviews per Month")

plt.tight_layout()
plt.savefig(fig_path / "05_reviews_per_month_by_occupancy_group.png", dpi=200)
plt.show()


## 6. Spatial Visualization

This plot uses longitude and latitude to visualize the geographic spread of listings. A sample is used to keep the plot readable.


In [ ]:
map_sample = df.sample(min(7000, len(df)), random_state=42)

plt.figure(figsize=(7, 6))

scatter = plt.scatter(
    map_sample["longitude"],
    map_sample["latitude"],
    c=map_sample["high_occupancy"],
    s=4,
    alpha=0.45
)

plt.title("NYC Airbnb Listings by Location and Occupancy Group")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.colorbar(scatter, label="High Occupancy")

plt.tight_layout()
plt.savefig(fig_path / "06_location_scatter_high_occupancy.png", dpi=200)
plt.show()


## 7. Correlation Analysis

We inspect correlations among selected numeric variables. This helps identify variables associated with the target and variables that may need special care in modeling.


In [ ]:
corr_cols = [
    "high_occupancy",
    "estimated_occupancy_l365d",
    "availability_365",
    "minimum_nights",
    "number_of_reviews",
    "number_of_reviews_ltm",
    "reviews_per_month",
    "review_scores_rating",
    "accommodates",
    "bedrooms",
    "beds",
    "host_acceptance_rate",
    "host_response_rate"
]

corr_cols = [c for c in corr_cols if c in df.columns]
corr = df[corr_cols].corr(numeric_only=True)
corr


In [ ]:
plt.figure(figsize=(9, 7))

im = plt.imshow(corr)
plt.title("Correlation Matrix for Selected Numeric Variables")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar(im)

plt.tight_layout()
plt.savefig(fig_path / "07_numeric_correlation_matrix.png", dpi=200)
plt.show()


## 8. PCA Preparation

For unsupervised learning, we use numeric variables related to host characteristics, location, capacity, availability, reviews, and review scores.

We do **not** include `estimated_occupancy_l365d` or `high_occupancy` in clustering, because those are outcome variables.


In [ ]:
cluster_features = [
    "hosts_time_as_host_years",
    "host_response_rate",
    "host_acceptance_rate",
    "host_listings_count",
    "host_total_listings_count",
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "maximum_nights",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "reviews_per_month",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "calculated_host_listings_count"
]

cluster_features = [c for c in cluster_features if c in df.columns]

X_cluster = df[cluster_features].copy()

for col in X_cluster.columns:
    X_cluster[col] = pd.to_numeric(X_cluster[col], errors="coerce")
    median_value = X_cluster[col].median()
    X_cluster[col] = X_cluster[col].fillna(0 if pd.isna(median_value) else median_value)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print("Number of clustering features:", len(cluster_features))


## 9. PCA

PCA gives us a two-dimensional view of listing structure before clustering.


In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_scores = pca.fit_transform(X_scaled)

print("Explained variance ratio:")
print(pca.explained_variance_ratio_)

df["pca1"] = pca_scores[:, 0]
df["pca2"] = pca_scores[:, 1]


In [ ]:
pca_sample = df.sample(min(7000, len(df)), random_state=42)

plt.figure(figsize=(7, 6))

scatter = plt.scatter(
    pca_sample["pca1"],
    pca_sample["pca2"],
    c=pca_sample["high_occupancy"],
    s=5,
    alpha=0.5
)

plt.title("PCA Projection Colored by High Occupancy")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.colorbar(scatter, label="High Occupancy")

plt.tight_layout()
plt.savefig(fig_path / "08_pca_high_occupancy.png", dpi=200)
plt.show()


## 10. K-Means Elbow Method

We test k values from 2 to 8 and inspect the elbow plot.


In [ ]:
ks = list(range(2, 9))
inertias = []

for k in ks:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

elbow_df = pd.DataFrame({
    "k": ks,
    "inertia": inertias
})

elbow_df


In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(elbow_df["k"], elbow_df["inertia"], marker="o")

plt.title("K-Means Elbow Plot")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")

plt.tight_layout()
plt.savefig(fig_path / "09_kmeans_elbow_plot.png", dpi=200)
plt.show()


## 11. Final K-Means Clustering

We use k = 4 because it provides an interpretable segmentation of listings without creating too many small clusters.


In [ ]:
k_final = 4

kmeans = KMeans(n_clusters=k_final, random_state=42, n_init=10)
df["listing_cluster"] = kmeans.fit_predict(X_scaled)

df["listing_cluster"].value_counts().sort_index()


In [ ]:
pca_sample = df.sample(min(7000, len(df)), random_state=42)

plt.figure(figsize=(7, 6))

scatter = plt.scatter(
    pca_sample["pca1"],
    pca_sample["pca2"],
    c=pca_sample["listing_cluster"],
    s=5,
    alpha=0.5
)

plt.title("PCA Projection Colored by K-Means Cluster")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.colorbar(scatter, label="Cluster")

plt.tight_layout()
plt.savefig(fig_path / "10_pca_kmeans_clusters.png", dpi=200)
plt.show()


## 12. Cluster Profiles

We summarize each cluster to interpret the listing segments discovered by K-Means.


In [ ]:
cluster_profile = (
    df.groupby("listing_cluster")
    .agg(
        listings=("id", "count"),
        high_occupancy_rate=("high_occupancy", "mean"),
        median_occupancy=("estimated_occupancy_l365d", "median"),
        median_availability_365=("availability_365", "median"),
        median_reviews=("number_of_reviews", "median"),
        median_reviews_per_month=("reviews_per_month", "median"),
        median_minimum_nights=("minimum_nights", "median"),
        median_accommodates=("accommodates", "median")
    )
    .reset_index()
    .sort_values("listing_cluster")
)

cluster_profile


In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    cluster_profile["listing_cluster"].astype(str),
    cluster_profile["high_occupancy_rate"]
)

plt.title("High-Occupancy Rate by K-Means Cluster")
plt.xlabel("K-Means Cluster")
plt.ylabel("High-Occupancy Rate")

for i, v in enumerate(cluster_profile["high_occupancy_rate"]):
    plt.text(i, v + 0.01, f"{v:.1%}", ha="center")

plt.tight_layout()
plt.savefig(fig_path / "11_cluster_high_occupancy_rate.png", dpi=200)
plt.show()


## 13. Save Outputs

We save the clustered dataset and supporting tables for later modeling and reporting.


In [ ]:
df.to_csv(processed_path / "nyc_airbnb_cleaned_with_clusters.csv", index=False)
cluster_profile.to_csv(processed_path / "cluster_profile.csv", index=False)
elbow_df.to_csv(processed_path / "kmeans_elbow_results.csv", index=False)

print("Saved clustered dataset and cluster profile.")


## Summary

Main takeaways:

1. The high-occupancy class represents a meaningful minority of listings, making binary classification appropriate.
2. Occupancy patterns vary by borough and room type, suggesting that location and listing type are important features.
3. Availability and review activity differ between high- and low-occupancy listings.
4. PCA provides a low-dimensional view of listing structure.
5. K-Means clustering identifies listing segments with different high-occupancy rates, suggesting that unsupervised learning captures meaningful structure in the data.
